In [29]:
!python -V

Python 3.9.25


In [30]:
import pandas as pd

In [31]:
import pickle

In [32]:
import seaborn as sns
import matplotlib.pyplot as plt

In [33]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [39]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi")

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1786731794129, experiment_id='5', last_update_time=1786731794129, lifecycle_stage='active', name='nyc-taxi', tags={}>

In [40]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [41]:
df_train = read_dataframe('./data/green_tripdata_2023-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2023-02.parquet')

In [42]:
len(df_train), len(df_val)

(65946, 62574)

In [43]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [44]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [45]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [46]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

2026/08/14 18:23:29 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4b9f562def5b46dda74abcdd7b627c53', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run funny-bee-62 at: http://127.0.0.1:5000/#/experiments/5/runs/4b9f562def5b46dda74abcdd7b627c53
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


6.03727552054262

In [47]:
with mlflow.start_run():

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2023-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2023-02.csv")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    # mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

🏃 View run bouncy-gnat-578 at: http://127.0.0.1:5000/#/experiments/5/runs/792e7847bae84dfabb17bf5a9acdb453
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [48]:
import xgboost as xgb

In [49]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [50]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [51]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=20
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [23]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|                                                                                                     | 0/50 [00:00<?, ?trial/s, best loss=?]

/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [15:46:40] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:8.27767                                                                                                                      
[1]	validation-rmse:7.47062                                                                                                                      
[2]	validation-rmse:6.85893                                                                                                                      
[3]	validation-rmse:6.40527                                                                                                                      
[4]	validation-rmse:6.06420                                                                                                                      
[5]	validation-rmse:5.82331                                                                                                                      
[6]	validation-rmse:5.64271                                                                                                 

/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [15:47:16] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:8.89937                                                                                                                      
[1]	validation-rmse:8.51572                                                                                                                      
[2]	validation-rmse:8.16724                                                                                                                      
[3]	validation-rmse:7.85221                                                                                                                      
[4]	validation-rmse:7.56920                                                                                                                      
[5]	validation-rmse:7.31246                                                                                                                      
[6]	validation-rmse:7.07900                                                                                                 

...

[2]	validation-rmse:8.21039                                                                                                                      
[3]	validation-rmse:7.90986                                                                                                                      
[4]	validation-rmse:7.63855                                                                                                                      
[5]	validation-rmse:7.39513                                                                                                                      
[6]	validation-rmse:7.17563                                                                                                                      
[7]	validation-rmse:6.98114                                                                                                                      
[8]	validation-rmse:6.80644                                                                                                 

/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [16:21:37] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:5.60010                                                                                                                      
[1]	validation-rmse:5.37436                                                                                                                      
[2]	validation-rmse:5.33318                                                                                                                      
[3]	validation-rmse:5.31648                                                                                                                      
[4]	validation-rmse:5.28212                                                                                                                      
[5]	validation-rmse:5.27591                                                                                                                      
[6]	validation-rmse:5.27175                                                                                                 

/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [16:21:46] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:8.97278                                                                                                                      
[1]	validation-rmse:8.65136                                                                                                                      
[2]	validation-rmse:8.35587                                                                                                                      
[3]	validation-rmse:8.08475                                                                                                                      
[4]	validation-rmse:7.83637                                                                                                                      
[5]	validation-rmse:7.60930                                                                                                                      
[6]	validation-rmse:7.40191                                                                                                 

In [52]:
mlflow.xgboost.autolog(disable=True)

In [53]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=100,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:25:36] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:8.73788
[1]	validation-rmse:8.22960
[2]	validation-rmse:7.78914
[3]	validation-rmse:7.40823
[4]	validation-rmse:7.08398
[5]	validation-rmse:6.80130
[6]	validation-rmse:6.56559
[7]	validation-rmse:6.35942
[8]	validation-rmse:6.18716
[9]	validation-rmse:6.04364
[10]	validation-rmse:5.91994
[11]	validation-rmse:5.81441
[12]	validation-rmse:5.72701
[13]	validation-rmse:5.65236
[14]	validation-rmse:5.58821
[15]	validation-rmse:5.53629
[16]	validation-rmse:5.49451
[17]	validation-rmse:5.45443
[18]	validation-rmse:5.42147
[19]	validation-rmse:5.39347
[20]	validation-rmse:5.37267
[21]	validation-rmse:5.35128
[22]	validation-rmse:5.33257
[23]	validation-rmse:5.31780
[24]	validation-rmse:5.30449
[25]	validation-rmse:5.29542
[26]	validation-rmse:5.28365
[27]	validation-rmse:5.27505
[28]	validation-rmse:5.26725
[29]	validation-rmse:5.26004
[30]	validation-rmse:5.25388
[31]	validation-rmse:5.24812
[32]	validation-rmse:5.24406
[33]	validation-rmse:5.24068
[34]	validation-rmse:5.2

2026/08/14 18:25:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:25:57] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/08/14 18:26:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run receptive-cow-599 at: http://127.0.0.1:5000/#/experiments/5/runs/1e3f13d90eba4d0f897b92aa2500a17f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [54]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        

🏃 View run adaptable-seal-97 at: http://127.0.0.1:5000/#/experiments/5/runs/20a1e974ff4a46be8810a971026a7089
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run painted-auk-257 at: http://127.0.0.1:5000/#/experiments/5/runs/e594e03acd5945d2a797e050319c15b8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run bouncy-worm-569 at: http://127.0.0.1:5000/#/experiments/5/runs/f1be7867870c485d8f87a131852d482c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


🏃 View run hilarious-grub-946 at: http://127.0.0.1:5000/#/experiments/5/runs/93934ee6e4524edfb6ef92c944483b54
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
